# Treino Baseline V3 — Comparativo

**Projeto:** CheckAI — Classificador binário de fake news em PT-BR  
**Notebook:** `treino_baseline_v3_comparativo.ipynb`  
**Fase:** 7

## Objetivo
Treinar e comparar modelos clássicos TF-IDF em três datasets:
- **V2 balanced** — baseline oficial da Fase anterior
- **V3 balanced** — candidato principal (V2 própria + FakeTrueBR + Fake.Br text_normalized)
- **V3 text_control** — ablação de viés de tamanho

## Modelos
- **Modelo A**: TF-IDF + Logistic Regression
- **Modelo B**: TF-IDF + LinearSVM
- **Modelo C**: TF-IDF + Multinomial Naive Bayes

## Regras obrigatórias
- Não alterar V1, V2 ou V3 datasets
- Não sobrescrever modelos antigos
- Salvar sempre com timestamp
- Campo de texto: `texto_principal_modelo` se existir, senão `texto_principal`
- Sem oversampling, sem BERTimbau, sem LLM
- `random_state=42`

## Referências anteriores
| Métrica | V1 | V2 balanced |
|---|---|---|
| Acurácia | 0.8625 | 0.9020 |
| F1 macro | 0.8614 | 0.9018 |
| ROC-AUC | 0.9359 | 0.9708 |

In [1]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')

# ── Detecção robusta da raiz do projeto ──────────────────────────────────────
def _find_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'dados').is_dir():
        return cwd
    if (cwd.parent / 'dados').is_dir():
        return cwd.parent
    raise FileNotFoundError(
        f"Pasta 'dados' não encontrada em '{cwd}' nem em '{cwd.parent}'.\n"
        "Execute o notebook a partir da raiz do projeto ou de src/."
    )

PROJECT_ROOT = _find_project_root()
DADOS_DIR    = PROJECT_ROOT / 'dados' / 'dataset_unificado' / 'final'
MODELOS_DIR  = PROJECT_ROOT / 'modelos'
MODELOS_DIR.mkdir(exist_ok=True)

SEED      = 42
TEST_SIZE = 0.2
TS        = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

print(f'CWD         : {Path.cwd()}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DADOS_DIR   : {DADOS_DIR}')
print(f'MODELOS_DIR : {MODELOS_DIR}')
print(f'TIMESTAMP   : {TS}')
print()
print('CSVs em DADOS_DIR:')
for csv in sorted(DADOS_DIR.glob('*.csv')):
    print(f'  {csv.name}')
print('\nImports OK.')

CWD         : C:\Users\offan\Desktop\ml-checkai\src
PROJECT_ROOT: C:\Users\offan\Desktop\ml-checkai
DADOS_DIR   : C:\Users\offan\Desktop\ml-checkai\dados\dataset_unificado\final
MODELOS_DIR : C:\Users\offan\Desktop\ml-checkai\modelos
TIMESTAMP   : 2026-05-27_00-29-53

CSVs em DADOS_DIR:
  dataset_final_treino_v1.csv
  dataset_final_treino_v2_balanced_2026-05-18_23-17-09.csv
  dataset_final_treino_v2_balanced_2026-05-18_23-31-48.csv
  dataset_final_treino_v2_balanced_2026-05-19_00-48-15.csv
  dataset_final_treino_v2_full_2026-05-18_23-17-09.csv
  dataset_final_treino_v2_full_2026-05-18_23-31-48.csv
  dataset_final_treino_v2_full_2026-05-19_00-48-15.csv
  dataset_final_treino_v2_size_matched_2026-05-18_23-42-41.csv
  dataset_final_treino_v3_balanced_2026-05-26_23-54-08.csv
  dataset_final_treino_v3_full_2026-05-26_23-54-08.csv
  dataset_final_treino_v3_text_control_2026-05-26_23-54-08.csv

Imports OK.


## Seção 1 — Detecção automática de datasets

In [2]:
def latest_match(pattern: str) -> Path:
    files = list(DADOS_DIR.glob(pattern))
    if not files:
        raise FileNotFoundError(f'Nenhum arquivo encontrado para padrão: {pattern}')
    return max(files, key=lambda p: p.stat().st_mtime)

PATH_V2_BALANCED      = latest_match('dataset_final_treino_v2_balanced_*.csv')
PATH_V3_BALANCED      = latest_match('dataset_final_treino_v3_balanced_*.csv')
PATH_V3_TEXT_CONTROL  = latest_match('dataset_final_treino_v3_text_control_*.csv')

print('Datasets detectados automaticamente:')
print(f'  V2 balanced     : {PATH_V2_BALANCED.name}')
print(f'  V3 balanced     : {PATH_V3_BALANCED.name}')
print(f'  V3 text_control : {PATH_V3_TEXT_CONTROL.name}')

Datasets detectados automaticamente:
  V2 balanced     : dataset_final_treino_v2_balanced_2026-05-19_00-48-15.csv
  V3 balanced     : dataset_final_treino_v3_balanced_2026-05-26_23-54-08.csv
  V3 text_control : dataset_final_treino_v3_text_control_2026-05-26_23-54-08.csv


## Seção 2 — Funções auxiliares e carregamento de datasets

In [3]:
def carregar_dataset(path: Path, nome: str) -> pd.DataFrame:
    """Carrega, valida e prepara um dataset para treino."""
    df = pd.read_csv(path, encoding='utf-8')
    print(f'\n{"-"*60}')
    print(f'Dataset: {nome}')
    print(f'Arquivo: {path.name}')
    print(f'Shape  : {df.shape}')

    # Valida colunas obrigatórias
    for col in ['label', 'texto_principal']:
        assert col in df.columns, f'Coluna obrigatória ausente: {col}'

    # Campo de texto preferencial
    if 'texto_principal_modelo' in df.columns:
        df['texto_treino'] = df['texto_principal_modelo']
        print('Campo de texto: texto_principal_modelo (preferencial)')
    else:
        df['texto_treino'] = df['texto_principal']
        print('Campo de texto: texto_principal (fallback — texto_principal_modelo ausente)')

    # Remove registros sem texto ou sem label
    n_antes = len(df)
    df = df.dropna(subset=['texto_treino', 'label'])
    df = df[df['texto_treino'].astype(str).str.strip() != '']
    n_depois = len(df)
    if n_antes != n_depois:
        print(f'AVISO: {n_antes - n_depois} registros removidos (sem texto ou label).')

    # Garante label binário 0/1
    df['label'] = df['label'].astype(int)
    assert set(df['label'].unique()).issubset({0, 1}), 'label deve ser binário 0/1'

    print(f'Registros válidos: {len(df)}')
    print(f'Distribuição label:')
    vc = df['label'].value_counts().sort_index()
    for lbl, cnt in vc.items():
        print(f'  label={lbl}: {cnt} ({100*cnt/len(df):.1f}%)')

    return df.reset_index(drop=True)


print('Função carregar_dataset definida.')

Função carregar_dataset definida.


In [4]:
df_v2_balanced     = carregar_dataset(PATH_V2_BALANCED,     'V2 balanced')
df_v3_balanced     = carregar_dataset(PATH_V3_BALANCED,     'V3 balanced')
df_v3_text_control = carregar_dataset(PATH_V3_TEXT_CONTROL, 'V3 text_control')

DATASETS = {
    'v2_balanced':     {'df': df_v2_balanced,     'path': PATH_V2_BALANCED},
    'v3_balanced':     {'df': df_v3_balanced,     'path': PATH_V3_BALANCED},
    'v3_text_control': {'df': df_v3_text_control, 'path': PATH_V3_TEXT_CONTROL},
}


------------------------------------------------------------
Dataset: V2 balanced
Arquivo: dataset_final_treino_v2_balanced_2026-05-19_00-48-15.csv
Shape  : (508, 11)
Campo de texto: texto_principal (fallback — texto_principal_modelo ausente)
Registros válidos: 508
Distribuição label:
  label=0: 254 (50.0%)
  label=1: 254 (50.0%)



------------------------------------------------------------
Dataset: V3 balanced
Arquivo: dataset_final_treino_v3_balanced_2026-05-26_23-54-08.csv
Shape  : (10340, 19)
Campo de texto: texto_principal_modelo (preferencial)
Registros válidos: 10340
Distribuição label:
  label=0: 5170 (50.0%)
  label=1: 5170 (50.0%)



------------------------------------------------------------
Dataset: V3 text_control
Arquivo: dataset_final_treino_v3_text_control_2026-05-26_23-54-08.csv
Shape  : (9050, 19)
Campo de texto: texto_principal_modelo (preferencial)
Registros válidos: 9050
Distribuição label:
  label=0: 4525 (50.0%)
  label=1: 4525 (50.0%)


## Seção 3 — Diagnóstico pré-treino

In [5]:
def diagnosticar_dataset(df: pd.DataFrame, nome: str):
    SEP = '=' * 65
    print(f'\n{SEP}')
    print(f'DIAGNÓSTICO — {nome}')
    print(SEP)
    print(f'Total de registros: {len(df)}')

    # Distribuição por label
    print('\n[Distribuição por label]')
    for lbl in sorted(df['label'].unique()):
        n = (df['label'] == lbl).sum()
        print(f'  label={lbl}: {n} ({100*n/len(df):.1f}%)')

    # dataset_origem
    if 'dataset_origem' in df.columns:
        print('\n[Distribuição por dataset_origem]')
        orig = df.groupby(['dataset_origem', 'label']).size().unstack(fill_value=0)
        orig['total'] = orig.sum(axis=1)
        print(orig.to_string())

    # origem_qualidade
    if 'origem_qualidade' in df.columns:
        print('\n[Distribuição por origem_qualidade]')
        oq = df.groupby(['origem_qualidade', 'label']).size().unstack(fill_value=0)
        oq['total'] = oq.sum(axis=1)
        print(oq.to_string())

    # Estatísticas de tamanho do texto_treino
    df = df.copy()
    df['_tam'] = df['texto_treino'].astype(str).str.len()
    print('\n[Tamanho do texto_treino por label (chars)]')
    hdr = f"  {'label':<8} {'n':>6} {'média':>8} {'mediana':>9} {'min':>6} {'max':>7}"
    print(hdr)
    print('  ' + '-' * (len(hdr) - 2))
    stats = {}
    for lbl in sorted(df['label'].unique()):
        sub = df[df['label'] == lbl]['_tam']
        med = sub.median()
        stats[lbl] = med
        print(f"  {lbl:<8} {len(sub):>6} {sub.mean():>8.0f} {med:>9.0f} {sub.min():>6.0f} {sub.max():>7.0f}")

    if 0 in stats and 1 in stats and stats[0] > 0:
        ratio = stats[1] / stats[0]
        aviso = '  *** VIÉS MODERADO ***' if ratio > 1.5 else ''
        print(f'  ratio mediana real(1)/fake(0): {ratio:.2f}x{aviso}')

    # faixa_tamanho_modelo
    if 'faixa_tamanho_modelo' in df.columns:
        print('\n[Distribuição por faixa_tamanho_modelo × label]')
        ft = df.groupby(['faixa_tamanho_modelo', 'label']).size().unstack(fill_value=0)
        ft['total'] = ft.sum(axis=1)
        print(ft.to_string())


for nome, info in DATASETS.items():
    diagnosticar_dataset(info['df'], nome)


DIAGNÓSTICO — v2_balanced
Total de registros: 508

[Distribuição por label]
  label=0: 254 (50.0%)
  label=1: 254 (50.0%)

[Distribuição por origem_qualidade]
label               0    1  total
origem_qualidade                 
ROTULO_ASSUMIDO     0  231    231
ROTULO_FORTE      254   23    277

[Tamanho do texto_treino por label (chars)]
  label         n    média   mediana    min     max
  -------------------------------------------------
  0           254       99        75     28     683
  1           254      245       213     34     497
  ratio mediana real(1)/fake(0): 2.84x  *** VIÉS MODERADO ***

DIAGNÓSTICO — v3_balanced
Total de registros: 10340

[Distribuição por label]
  label=0: 5170 (50.0%)
  label=1: 5170 (50.0%)

[Distribuição por dataset_origem]
label                      0     1  total
dataset_origem                           
FAKEBR_TEXT_NORMALIZED  3436  3582   7018
FAKETRUEBR              1492  1340   2832
V2_PROPRIA               242   248    490

[Distribuição po

## Seção 4 — Split treino/teste estratificado

In [6]:
def fazer_split(df: pd.DataFrame, nome: str):
    X = df['texto_treino'].astype(str).values
    y = df['label'].values
    idx = df.index.values

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, idx,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    print(f'\n[Split — {nome}]')
    print(f'  Treino : {len(X_train)} | label=0: {(y_train==0).sum()} | label=1: {(y_train==1).sum()}')
    print(f'  Teste  : {len(X_test)}  | label=0: {(y_test==0).sum()}  | label=1: {(y_test==1).sum()}')

    # Distribuição de dataset_origem no treino/teste (V3 apenas)
    if 'dataset_origem' in df.columns:
        df_train = df.loc[idx_train]
        df_test  = df.loc[idx_test]
        print('  [dataset_origem no treino]')
        for orig, cnt in df_train['dataset_origem'].value_counts().items():
            print(f'    {orig}: {cnt}')
        print('  [dataset_origem no teste]')
        for orig, cnt in df_test['dataset_origem'].value_counts().items():
            print(f'    {orig}: {cnt}')

    return {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'idx_train': idx_train, 'idx_test': idx_test,
    }


print('Realizando splits...')
for nome, info in DATASETS.items():
    info['split'] = fazer_split(info['df'], nome)
print('\nSplits concluídos.')

Realizando splits...

[Split — v2_balanced]
  Treino : 406 | label=0: 203 | label=1: 203
  Teste  : 102  | label=0: 51  | label=1: 51

[Split — v3_balanced]
  Treino : 8272 | label=0: 4136 | label=1: 4136
  Teste  : 2068  | label=0: 1034  | label=1: 1034
  [dataset_origem no treino]
    FAKEBR_TEXT_NORMALIZED: 5631
    FAKETRUEBR: 2260
    V2_PROPRIA: 381
  [dataset_origem no teste]
    FAKEBR_TEXT_NORMALIZED: 1387
    FAKETRUEBR: 572
    V2_PROPRIA: 109

[Split — v3_text_control]
  Treino : 7240 | label=0: 3620 | label=1: 3620
  Teste  : 1810  | label=0: 905  | label=1: 905
  [dataset_origem no treino]
    FAKEBR_TEXT_NORMALIZED: 5546
    FAKETRUEBR: 1447
    V2_PROPRIA: 247
  [dataset_origem no teste]
    FAKEBR_TEXT_NORMALIZED: 1381
    FAKETRUEBR: 366
    V2_PROPRIA: 63

Splits concluídos.


## Seção 5 — Pipelines de modelos

### Parâmetros TF-IDF (padrão)
- `lowercase=True`, `strip_accents='unicode'`, `ngram_range=(1,2)`
- `min_df=2`, `max_df=0.9`, `max_features=50000`, `sublinear_tf=True`

### Parâmetros TF-IDF (para MultinomialNB)
- Idênticos, mas `sublinear_tf=False` (NB pressupõe contagens)

### Modelos
- **A** – LogisticRegression (`liblinear`, `class_weight='balanced'`)
- **B** – LinearSVC (`class_weight='balanced'`), calibrado para proba
- **C** – MultinomialNB

In [7]:
TFIDF_PARAMS = dict(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    max_features=50000,
    sublinear_tf=True,
)

TFIDF_PARAMS_NB = {**TFIDF_PARAMS, 'sublinear_tf': False}


def make_logreg() -> Pipeline:
    return Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LogisticRegression(
            solver='liblinear',
            class_weight='balanced',
            max_iter=1000,
            random_state=SEED,
        )),
    ])


def make_linearsvm() -> Pipeline:
    # CalibratedClassifierCV permite predict_proba para LinearSVC
    svm = CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', random_state=SEED, max_iter=2000),
        cv=3,
    )
    return Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   svm),
    ])


def make_multinomialnb() -> Pipeline:
    return Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS_NB)),
        ('clf',   MultinomialNB()),
    ])


MODELOS_FACTORY = {
    'logreg': make_logreg,
    'svm':    make_linearsvm,
    'nb':     make_multinomialnb,
}

print('Factories definidas: logreg, svm (calibrado), nb')

Factories definidas: logreg, svm (calibrado), nb


## Seção 6 — Treinamento (3 datasets × 3 modelos)

In [8]:
RESULTADOS = {}  # chave: 'dataset_modelo'

print('Iniciando treinamento...\n')

for ds_nome, ds_info in DATASETS.items():
    split = ds_info['split']
    X_train = split['X_train']
    y_train = split['y_train']

    for mod_nome, factory in MODELOS_FACTORY.items():
        chave = f'{ds_nome}__{mod_nome}'
        print(f'Treinando: {chave} ...')
        pipe = factory()
        pipe.fit(X_train, y_train)

        n_features = pipe.named_steps['tfidf'].get_feature_names_out().shape[0]
        print(f'  Features TF-IDF: {n_features}')

        RESULTADOS[chave] = {
            'pipe':      pipe,
            'ds_nome':   ds_nome,
            'mod_nome':  mod_nome,
            'n_features': n_features,
        }

print(f'\nTreinamento concluído. {len(RESULTADOS)} modelos gerados.')

Iniciando treinamento...

Treinando: v2_balanced__logreg ...
  Features TF-IDF: 1935
Treinando: v2_balanced__svm ...
  Features TF-IDF: 1935
Treinando: v2_balanced__nb ...


  Features TF-IDF: 1935
Treinando: v3_balanced__logreg ...


  Features TF-IDF: 50000
Treinando: v3_balanced__svm ...


  Features TF-IDF: 50000
Treinando: v3_balanced__nb ...


  Features TF-IDF: 50000
Treinando: v3_text_control__logreg ...


  Features TF-IDF: 50000
Treinando: v3_text_control__svm ...


  Features TF-IDF: 50000
Treinando: v3_text_control__nb ...


  Features TF-IDF: 50000

Treinamento concluído. 9 modelos gerados.


## Seção 7 — Avaliação de modelos

In [9]:
def avaliar_pipeline(pipe: Pipeline, X_test, y_test) -> dict:
    y_pred = pipe.predict(X_test)

    metricas = {
        'accuracy':          round(accuracy_score(y_test, y_pred), 4),
        'precision_macro':   round(precision_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'recall_macro':      round(recall_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'f1_macro':          round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4),
        'precision_label_0': round(precision_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0), 4),
        'recall_label_0':    round(recall_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0), 4),
        'f1_label_0':        round(f1_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0), 4),
        'precision_label_1': round(precision_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0), 4),
        'recall_label_1':    round(recall_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0), 4),
        'f1_label_1':        round(f1_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0), 4),
        'roc_auc':           None,
        'pr_auc':            None,
    }

    # ROC-AUC e PR-AUC (predict_proba se disponível)
    y_score = None
    if hasattr(pipe, 'predict_proba'):
        try:
            y_score = pipe.predict_proba(X_test)[:, 1]
        except Exception:
            pass
    if y_score is not None:
        metricas['roc_auc'] = round(roc_auc_score(y_test, y_score), 4)
        metricas['pr_auc']  = round(average_precision_score(y_test, y_score), 4)

    return metricas, y_pred, y_score


def imprimir_avaliacao(nome: str, metricas: dict, y_test, y_pred):
    SEP = '-' * 60
    print(f'\n{SEP}')
    print(f'Métricas — {nome}')
    print(SEP)
    for k, v in metricas.items():
        val = f'{v:.4f}' if v is not None else 'N/A'
        print(f'  {k:<22}: {val}')

    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=['Real 0 (fake)', 'Real 1 (real)'],
        columns=['Pred 0 (fake)', 'Pred 1 (real)'],
    )
    print('\nMatriz de confusão:')
    print(cm_df.to_string())
    print('\nClassification report:')
    print(classification_report(
        y_test, y_pred,
        target_names=['fake/misleading (0)', 'real/verdadeiro (1)'],
        zero_division=0,
    ))


print('Funções de avaliação definidas.')

Funções de avaliação definidas.


In [10]:
print('Avaliando todos os modelos...\n')

for chave, res in RESULTADOS.items():
    ds_nome  = res['ds_nome']
    split    = DATASETS[ds_nome]['split']
    X_test   = split['X_test']
    y_test   = split['y_test']

    metricas, y_pred, y_score = avaliar_pipeline(res['pipe'], X_test, y_test)
    imprimir_avaliacao(chave, metricas, y_test, y_pred)

    res['metricas'] = metricas
    res['y_pred']   = y_pred
    res['y_score']  = y_score

print('\nAvaliação concluída.')

Avaliando todos os modelos...


------------------------------------------------------------
Métricas — v2_balanced__logreg
------------------------------------------------------------
  accuracy              : 0.9020
  precision_macro       : 0.9026
  recall_macro          : 0.9020
  f1_macro              : 0.9019
  precision_label_0     : 0.8868
  recall_label_0        : 0.9216
  f1_label_0            : 0.9038
  precision_label_1     : 0.9184
  recall_label_1        : 0.8824
  f1_label_1            : 0.9000
  roc_auc               : 0.9727
  pr_auc                : 0.9721

Matriz de confusão:
               Pred 0 (fake)  Pred 1 (real)
Real 0 (fake)             47              4
Real 1 (real)              6             45

Classification report:
                     precision    recall  f1-score   support

fake/misleading (0)       0.89      0.92      0.90        51
real/verdadeiro (1)       0.92      0.88      0.90        51

           accuracy                           0.90       


------------------------------------------------------------
Métricas — v3_balanced__logreg
------------------------------------------------------------
  accuracy              : 0.9400
  precision_macro       : 0.9400
  recall_macro          : 0.9400
  f1_macro              : 0.9400
  precision_label_0     : 0.9400
  recall_label_0        : 0.9400
  f1_label_0            : 0.9400
  precision_label_1     : 0.9400
  recall_label_1        : 0.9400
  f1_label_1            : 0.9400
  roc_auc               : 0.9787
  pr_auc                : 0.9805

Matriz de confusão:
               Pred 0 (fake)  Pred 1 (real)
Real 0 (fake)            972             62
Real 1 (real)             62            972

Classification report:
                     precision    recall  f1-score   support

fake/misleading (0)       0.94      0.94      0.94      1034
real/verdadeiro (1)       0.94      0.94      0.94      1034

           accuracy                           0.94      2068
          macro avg       0


------------------------------------------------------------
Métricas — v3_balanced__svm
------------------------------------------------------------
  accuracy              : 0.9463
  precision_macro       : 0.9463
  recall_macro          : 0.9463
  f1_macro              : 0.9463
  precision_label_0     : 0.9450
  recall_label_0        : 0.9478
  f1_label_0            : 0.9464
  precision_label_1     : 0.9476
  recall_label_1        : 0.9449
  f1_label_1            : 0.9462
  roc_auc               : 0.9845
  pr_auc                : 0.9847

Matriz de confusão:
               Pred 0 (fake)  Pred 1 (real)
Real 0 (fake)            980             54
Real 1 (real)             57            977

Classification report:
                     precision    recall  f1-score   support

fake/misleading (0)       0.95      0.95      0.95      1034
real/verdadeiro (1)       0.95      0.94      0.95      1034

           accuracy                           0.95      2068
          macro avg       0.95


------------------------------------------------------------
Métricas — v3_balanced__nb
------------------------------------------------------------
  accuracy              : 0.8598
  precision_macro       : 0.8756
  recall_macro          : 0.8598
  f1_macro              : 0.8583
  precision_label_0     : 0.9526
  recall_label_0        : 0.7573
  f1_label_0            : 0.8438
  precision_label_1     : 0.7986
  recall_label_1        : 0.9623
  f1_label_1            : 0.8728
  roc_auc               : 0.9611
  pr_auc                : 0.9625

Matriz de confusão:
               Pred 0 (fake)  Pred 1 (real)
Real 0 (fake)            783            251
Real 1 (real)             39            995

Classification report:
                     precision    recall  f1-score   support

fake/misleading (0)       0.95      0.76      0.84      1034
real/verdadeiro (1)       0.80      0.96      0.87      1034

           accuracy                           0.86      2068
          macro avg       0.88 


------------------------------------------------------------
Métricas — v3_text_control__logreg
------------------------------------------------------------
  accuracy              : 0.9243
  precision_macro       : 0.9243
  recall_macro          : 0.9243
  f1_macro              : 0.9243
  precision_label_0     : 0.9267
  recall_label_0        : 0.9215
  f1_label_0            : 0.9241
  precision_label_1     : 0.9220
  recall_label_1        : 0.9271
  f1_label_1            : 0.9245
  roc_auc               : 0.9749
  pr_auc                : 0.9749

Matriz de confusão:
               Pred 0 (fake)  Pred 1 (real)
Real 0 (fake)            834             71
Real 1 (real)             66            839

Classification report:
                     precision    recall  f1-score   support

fake/misleading (0)       0.93      0.92      0.92       905
real/verdadeiro (1)       0.92      0.93      0.92       905

           accuracy                           0.92      1810
          macro avg    


------------------------------------------------------------
Métricas — v3_text_control__svm
------------------------------------------------------------
  accuracy              : 0.9337
  precision_macro       : 0.9338
  recall_macro          : 0.9337
  f1_macro              : 0.9337
  precision_label_0     : 0.9415
  recall_label_0        : 0.9249
  f1_label_0            : 0.9331
  precision_label_1     : 0.9262
  recall_label_1        : 0.9425
  f1_label_1            : 0.9343
  roc_auc               : 0.9816
  pr_auc                : 0.9819

Matriz de confusão:
               Pred 0 (fake)  Pred 1 (real)
Real 0 (fake)            837             68
Real 1 (real)             52            853

Classification report:
                     precision    recall  f1-score   support

fake/misleading (0)       0.94      0.92      0.93       905
real/verdadeiro (1)       0.93      0.94      0.93       905

           accuracy                           0.93      1810
          macro avg       


------------------------------------------------------------
Métricas — v3_text_control__nb
------------------------------------------------------------
  accuracy              : 0.8735
  precision_macro       : 0.8810
  recall_macro          : 0.8735
  f1_macro              : 0.8729
  precision_label_0     : 0.9344
  recall_label_0        : 0.8033
  f1_label_0            : 0.8639
  precision_label_1     : 0.8275
  recall_label_1        : 0.9436
  f1_label_1            : 0.8818
  roc_auc               : 0.9477
  pr_auc                : 0.9477

Matriz de confusão:
               Pred 0 (fake)  Pred 1 (real)
Real 0 (fake)            727            178
Real 1 (real)             51            854

Classification report:
                     precision    recall  f1-score   support

fake/misleading (0)       0.93      0.80      0.86       905
real/verdadeiro (1)       0.83      0.94      0.88       905

           accuracy                           0.87      1810
          macro avg       0

## Seção 8 — Análise por dataset_origem (V3)

In [11]:
registros_por_origem = []

def analisar_por_origem(chave: str, res: dict):
    ds_nome = res['ds_nome']
    df      = DATASETS[ds_nome]['df']
    split   = DATASETS[ds_nome]['split']

    if 'dataset_origem' not in df.columns:
        print(f'  [{chave}] dataset_origem ausente — análise por origem não aplicável.')
        return

    idx_test  = split['idx_test']
    df_test   = df.loc[idx_test].copy()
    df_test['_pred'] = res['y_pred']

    print(f'\n{"-"*60}')
    print(f'Análise por dataset_origem — {chave}')
    print(f'{"-"*60}')

    for origem in sorted(df_test['dataset_origem'].dropna().unique()):
        sub = df_test[df_test['dataset_origem'] == origem]
        if len(sub) == 0:
            continue
        acc = accuracy_score(sub['label'], sub['_pred'])
        f1  = f1_score(sub['label'], sub['_pred'], average='macro', zero_division=0)
        p0  = precision_score(sub['label'], sub['_pred'], pos_label=0, average='binary', zero_division=0)
        r0  = recall_score(sub['label'], sub['_pred'], pos_label=0, average='binary', zero_division=0)
        f0  = f1_score(sub['label'], sub['_pred'], pos_label=0, average='binary', zero_division=0)
        p1  = precision_score(sub['label'], sub['_pred'], pos_label=1, average='binary', zero_division=0)
        r1  = recall_score(sub['label'], sub['_pred'], pos_label=1, average='binary', zero_division=0)
        f1l = f1_score(sub['label'], sub['_pred'], pos_label=1, average='binary', zero_division=0)

        print(f'  {origem} (n={len(sub)})')
        print(f'    acc={acc:.4f}  f1_macro={f1:.4f}')
        print(f'    label=0: p={p0:.3f} r={r0:.3f} f1={f0:.3f}')
        print(f'    label=1: p={p1:.3f} r={r1:.3f} f1={f1l:.3f}')

        registros_por_origem.append({
            'chave_modelo':  chave,
            'dataset_treino': ds_nome,
            'modelo':        res['mod_nome'],
            'dataset_origem': origem,
            'n_teste':       len(sub),
            'accuracy':      round(acc, 4),
            'f1_macro':      round(f1, 4),
            'precision_label_0': round(p0, 4),
            'recall_label_0':    round(r0, 4),
            'f1_label_0':        round(f0, 4),
            'precision_label_1': round(p1, 4),
            'recall_label_1':    round(r1, 4),
            'f1_label_1':        round(f1l, 4),
        })


print('Análise por origem:\n')
for chave, res in RESULTADOS.items():
    if res['ds_nome'].startswith('v3'):
        analisar_por_origem(chave, res)

Análise por origem:


------------------------------------------------------------
Análise por dataset_origem — v3_balanced__logreg
------------------------------------------------------------
  FAKEBR_TEXT_NORMALIZED (n=1387)
    acc=0.9474  f1_macro=0.9472
    label=0: p=0.963 r=0.927 f1=0.944
    label=1: p=0.934 r=0.967 f1=0.950
  FAKETRUEBR (n=572)
    acc=0.9615  f1_macro=0.9612
    label=0: p=0.971 r=0.959 f1=0.965
    label=1: p=0.950 r=0.965 f1=0.958
  V2_PROPRIA (n=109)
    acc=0.7339  f1_macro=0.7247
    label=0: p=0.633 r=1.000 f1=0.775
    label=1: p=1.000 r=0.508 f1=0.674

------------------------------------------------------------
Análise por dataset_origem — v3_balanced__svm
------------------------------------------------------------
  FAKEBR_TEXT_NORMALIZED (n=1387)
    acc=0.9495  f1_macro=0.9494
    label=0: p=0.959 r=0.936 f1=0.947
    label=1: p=0.941 r=0.962 f1=0.952
  FAKETRUEBR (n=572)
    acc=0.9738  f1_macro=0.9735
    label=0: p=0.984 r=0.968 f1=0.976
    l

  V2_PROPRIA (n=63)
    acc=0.6190  f1_macro=0.5891
    label=0: p=0.314 r=1.000 f1=0.478
    label=1: p=1.000 r=0.538 f1=0.700

------------------------------------------------------------
Análise por dataset_origem — v3_text_control__svm
------------------------------------------------------------
  FAKEBR_TEXT_NORMALIZED (n=1381)
    acc=0.9457  f1_macro=0.9456
    label=0: p=0.963 r=0.925 f1=0.944
    label=1: p=0.930 r=0.966 f1=0.947
  FAKETRUEBR (n=366)
    acc=0.9344  f1_macro=0.9334
    label=0: p=0.960 r=0.924 f1=0.942
    label=1: p=0.902 r=0.949 f1=0.925
  V2_PROPRIA (n=63)
    acc=0.6667  f1_macro=0.6204
    label=0: p=0.333 r=0.909 f1=0.488
    label=1: p=0.970 r=0.615 f1=0.753

------------------------------------------------------------
Análise por dataset_origem — v3_text_control__nb
------------------------------------------------------------
  FAKEBR_TEXT_NORMALIZED (n=1381)
    acc=0.8704  f1_macro=0.8692
    label=0: p=0.945 r=0.784 f1=0.857
    label=1: p=0.818 r=0

## Seção 9 — Análise por faixa de tamanho

In [12]:
registros_por_tamanho = []

def analisar_por_tamanho(chave: str, res: dict):
    ds_nome = res['ds_nome']
    df      = DATASETS[ds_nome]['df']
    split   = DATASETS[ds_nome]['split']

    col_faixa = None
    if 'faixa_tamanho_modelo' in df.columns:
        col_faixa = 'faixa_tamanho_modelo'
    elif 'faixa_tamanho' in df.columns:
        col_faixa = 'faixa_tamanho'

    if col_faixa is None:
        print(f'  [{chave}] Coluna de faixa ausente — análise por tamanho não aplicável.')
        return

    idx_test  = split['idx_test']
    df_test   = df.loc[idx_test].copy()
    df_test['_pred'] = res['y_pred']

    print(f'\n{"-"*60}')
    print(f'Análise por faixa de tamanho — {chave}')
    print(f'{"-"*60}')

    for faixa in sorted(df_test[col_faixa].dropna().unique()):
        sub = df_test[df_test[col_faixa] == faixa]
        if len(sub) < 2:
            continue
        acc = accuracy_score(sub['label'], sub['_pred'])
        f1  = f1_score(sub['label'], sub['_pred'], average='macro', zero_division=0)
        erros = (sub['label'] != sub['_pred']).sum()
        print(f'  {faixa:<20} n={len(sub):>5}  acc={acc:.3f}  f1_macro={f1:.3f}  erros={erros}')

        registros_por_tamanho.append({
            'chave_modelo':  chave,
            'dataset_treino': ds_nome,
            'modelo':        res['mod_nome'],
            'faixa_tamanho': faixa,
            'n_teste':       len(sub),
            'accuracy':      round(acc, 4),
            'f1_macro':      round(f1, 4),
            'erros':         int(erros),
        })


print('Análise por tamanho:\n')
for chave, res in RESULTADOS.items():
    if res['ds_nome'].startswith('v3'):
        analisar_por_tamanho(chave, res)

Análise por tamanho:


------------------------------------------------------------
Análise por faixa de tamanho — v3_balanced__logreg
------------------------------------------------------------
  curto                n=  294  acc=0.881  f1_macro=0.787  erros=35
  longo                n=  124  acc=0.960  f1_macro=0.924  erros=5
  medio                n= 1618  acc=0.948  f1_macro=0.948  erros=84
  muito_longo          n=   32  acc=1.000  f1_macro=1.000  erros=0

------------------------------------------------------------
Análise por faixa de tamanho — v3_balanced__svm
------------------------------------------------------------
  curto                n=  294  acc=0.895  f1_macro=0.823  erros=31
  longo                n=  124  acc=1.000  f1_macro=1.000  erros=0
  medio                n= 1618  acc=0.951  f1_macro=0.950  erros=80
  muito_longo          n=   32  acc=1.000  f1_macro=1.000  erros=0

------------------------------------------------------------
Análise por faixa de tamanho — 

## Seção 10 — Análise de erros do melhor modelo V3 balanced

In [13]:
# Identificar melhor modelo na V3 balanced (por f1_macro)
chaves_v3_bal = [k for k in RESULTADOS if k.startswith('v3_balanced__')]
melhor_v3_bal = max(chaves_v3_bal, key=lambda k: RESULTADOS[k]['metricas']['f1_macro'])

print(f'Melhor modelo V3 balanced: {melhor_v3_bal}')
print(f'  f1_macro = {RESULTADOS[melhor_v3_bal]["metricas"]["f1_macro"]}')

res      = RESULTADOS[melhor_v3_bal]
df       = DATASETS['v3_balanced']['df']
split    = DATASETS['v3_balanced']['split']
idx_test = split['idx_test']

df_erros = df.loc[idx_test].copy()
df_erros['label_predito'] = res['y_pred']
df_erros = df_erros[df_erros['label'] != df_erros['label_predito']].copy()
df_erros = df_erros.rename(columns={'label': 'label_real'})

# Adiciona score se disponível
if res['y_score'] is not None:
    score_test = res['y_score']
    score_series = pd.Series(score_test, index=idx_test)
    df_erros['score_prob_label1'] = df_erros.index.map(score_series)

# Seleciona colunas de interesse
cols_saida = ['texto_treino', 'label_real', 'label_predito']
for col in ['dataset_origem', 'fonte_dataset', 'origem_qualidade',
            'tamanho_chars_modelo', 'faixa_tamanho_modelo', 'score_prob_label1']:
    if col in df_erros.columns:
        cols_saida.append(col)

df_erros_out = df_erros[[c for c in cols_saida if c in df_erros.columns]]

print(f'\nTotal de erros (FP + FN): {len(df_erros_out)}')
print(f'  FP (real classificado como fake): {((df_erros["label_real"]==1) & (df_erros["label_predito"]==0)).sum()}')
print(f'  FN (fake classificado como real): {((df_erros["label_real"]==0) & (df_erros["label_predito"]==1)).sum()}')

if 'dataset_origem' in df_erros_out.columns:
    print('\nErros por dataset_origem:')
    print(df_erros_out['dataset_origem'].value_counts().to_string())

if 'faixa_tamanho_modelo' in df_erros_out.columns:
    print('\nErros por faixa_tamanho_modelo:')
    print(df_erros_out['faixa_tamanho_modelo'].value_counts().to_string())

# Salva
path_erros = DADOS_DIR / f'erros_modelo_v3_{TS}.csv'
df_erros_out.to_csv(path_erros, index=False, encoding='utf-8')
print(f'\nErros salvos em: {path_erros.name}')

Melhor modelo V3 balanced: v3_balanced__svm
  f1_macro = 0.9463

Total de erros (FP + FN): 111
  FP (real classificado como fake): 57
  FN (fake classificado como real): 54

Erros por dataset_origem:
dataset_origem
FAKEBR_TEXT_NORMALIZED    70
V2_PROPRIA                26
FAKETRUEBR                15

Erros por faixa_tamanho_modelo:
faixa_tamanho_modelo
medio    80
curto    31

Erros salvos em: erros_modelo_v3_2026-05-27_00-29-53.csv


## Seção 11 — Salvar relatórios CSV

In [14]:
# ── Relatório principal de métricas ─────────────────────────────────────────
registros_metricas = []

for chave, res in RESULTADOS.items():
    ds_nome  = res['ds_nome']
    df       = DATASETS[ds_nome]['df']
    split    = DATASETS[ds_nome]['split']
    m        = res['metricas']

    registros_metricas.append({
        'dataset_treino':    ds_nome,
        'arquivo_dataset':   DATASETS[ds_nome]['path'].name,
        'modelo':            res['mod_nome'],
        'total_registros':   len(df),
        'total_treino':      len(split['X_train']),
        'total_teste':       len(split['X_test']),
        'n_features_tfidf':  res['n_features'],
        'accuracy':          m['accuracy'],
        'precision_macro':   m['precision_macro'],
        'recall_macro':      m['recall_macro'],
        'f1_macro':          m['f1_macro'],
        'precision_label_0': m['precision_label_0'],
        'recall_label_0':    m['recall_label_0'],
        'f1_label_0':        m['f1_label_0'],
        'precision_label_1': m['precision_label_1'],
        'recall_label_1':    m['recall_label_1'],
        'f1_label_1':        m['f1_label_1'],
        'roc_auc':           m['roc_auc'],
        'pr_auc':            m['pr_auc'],
        'observacao':        f'campo_texto=texto_principal_modelo' if 'texto_principal_modelo' in df.columns
                             else 'campo_texto=texto_principal (fallback)',
    })

df_metricas = pd.DataFrame(registros_metricas)
path_metricas = DADOS_DIR / f'metricas_modelos_v3_{TS}.csv'
df_metricas.to_csv(path_metricas, index=False, encoding='utf-8')
print(f'Métricas principais salvas: {path_metricas.name}')

# ── Métricas por origem ──────────────────────────────────────────────────────
if registros_por_origem:
    df_orig = pd.DataFrame(registros_por_origem)
    path_orig = DADOS_DIR / f'metricas_por_origem_v3_{TS}.csv'
    df_orig.to_csv(path_orig, index=False, encoding='utf-8')
    print(f'Métricas por origem salvas: {path_orig.name}')

# ── Métricas por tamanho ─────────────────────────────────────────────────────
if registros_por_tamanho:
    df_tam = pd.DataFrame(registros_por_tamanho)
    path_tam = DADOS_DIR / f'metricas_por_tamanho_v3_{TS}.csv'
    df_tam.to_csv(path_tam, index=False, encoding='utf-8')
    print(f'Métricas por tamanho salvas: {path_tam.name}')

print('\nRelatórios CSV gerados com sucesso.')

Métricas principais salvas: metricas_modelos_v3_2026-05-27_00-29-53.csv
Métricas por origem salvas: metricas_por_origem_v3_2026-05-27_00-29-53.csv
Métricas por tamanho salvas: metricas_por_tamanho_v3_2026-05-27_00-29-53.csv

Relatórios CSV gerados com sucesso.


## Seção 12 — Salvar modelos com timestamp

In [15]:
# Nomes apenas para V3 (V2 já tem modelos salvos)
NOMES_ARQUIVO = {
    'v3_balanced__logreg': 'baseline_tfidf_logreg_v3_balanced',
    'v3_balanced__svm':    'baseline_tfidf_svm_v3_balanced',
    'v3_balanced__nb':     'baseline_tfidf_nb_v3_balanced',
    'v3_text_control__logreg': 'baseline_tfidf_logreg_v3_text_control',
    'v3_text_control__svm':    'baseline_tfidf_svm_v3_text_control',
    'v3_text_control__nb':     'baseline_tfidf_nb_v3_text_control',
}


def salvar_modelo(chave: str, res: dict):
    nome_base = NOMES_ARQUIVO.get(chave)
    if nome_base is None:
        return None, None

    ds_nome  = res['ds_nome']
    df       = DATASETS[ds_nome]['df']
    split    = DATASETS[ds_nome]['split']
    m        = res['metricas']

    nome_com_ts = f'{nome_base}_{TS}'
    path_joblib = MODELOS_DIR / f'{nome_com_ts}.joblib'
    path_meta   = MODELOS_DIR / f'{nome_com_ts}_meta.json'

    joblib.dump(res['pipe'], path_joblib)

    campo_texto = 'texto_principal_modelo' if 'texto_principal_modelo' in df.columns else 'texto_principal'
    tfidf_p = {k: list(v) if isinstance(v, tuple) else v for k, v in TFIDF_PARAMS.items()}

    meta = {
        'nome_modelo':          nome_com_ts,
        'data_treino':          TS.replace('_', ' ').replace('-', '-'),
        'dataset_treino':       ds_nome,
        'arquivo_dataset':      DATASETS[ds_nome]['path'].name,
        'campo_texto':          campo_texto,
        'random_state':         SEED,
        'test_size':            TEST_SIZE,
        'n_total':              len(df),
        'n_treino':             len(split['X_train']),
        'n_teste':              len(split['X_test']),
        'n_features_tfidf':     res['n_features'],
        'tfidf_params':         tfidf_p,
        'modelo_tipo':          res['mod_nome'],
        'metricas_teste':       m,
        'observacoes':          [
            'V3 usa texto_principal_modelo para reduzir viés de tamanho.',
            'Sem oversampling. Sem BERTimbau. Sem LLM.',
            'Split estratificado por label com random_state=42.',
            'LinearSVM calibrado com CalibratedClassifierCV(cv=3) para predict_proba.',
            'MultinomialNB usa sublinear_tf=False para preservar semântica de contagens.',
        ],
    }

    with open(path_meta, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f'  Salvo: {path_joblib.name}')
    print(f'  Salvo: {path_meta.name}')
    return path_joblib, path_meta


print('Salvando modelos V3...\n')
for chave, res in RESULTADOS.items():
    if chave in NOMES_ARQUIVO:
        salvar_modelo(chave, res)

print('\nModelos V3 salvos.')

Salvando modelos V3...



  Salvo: baseline_tfidf_logreg_v3_balanced_2026-05-27_00-29-53.joblib
  Salvo: baseline_tfidf_logreg_v3_balanced_2026-05-27_00-29-53_meta.json


  Salvo: baseline_tfidf_svm_v3_balanced_2026-05-27_00-29-53.joblib
  Salvo: baseline_tfidf_svm_v3_balanced_2026-05-27_00-29-53_meta.json


  Salvo: baseline_tfidf_nb_v3_balanced_2026-05-27_00-29-53.joblib
  Salvo: baseline_tfidf_nb_v3_balanced_2026-05-27_00-29-53_meta.json


  Salvo: baseline_tfidf_logreg_v3_text_control_2026-05-27_00-29-53.joblib
  Salvo: baseline_tfidf_logreg_v3_text_control_2026-05-27_00-29-53_meta.json


  Salvo: baseline_tfidf_svm_v3_text_control_2026-05-27_00-29-53.joblib
  Salvo: baseline_tfidf_svm_v3_text_control_2026-05-27_00-29-53_meta.json


  Salvo: baseline_tfidf_nb_v3_text_control_2026-05-27_00-29-53.joblib
  Salvo: baseline_tfidf_nb_v3_text_control_2026-05-27_00-29-53_meta.json

Modelos V3 salvos.


## Seção 13 — Relatório final comparativo

In [16]:
V1_REF = {'accuracy': 0.8625, 'f1_macro': 0.8614, 'roc_auc': 0.9359}
V2_REF = {'accuracy': 0.9020, 'f1_macro': 0.9018, 'roc_auc': 0.9708}

SEP = '=' * 72
print(SEP)
print('RELATÓRIO FINAL — TREINO BASELINE V3 COMPARATIVO')
print(SEP)

# Arquivos usados
print('\n[Datasets utilizados]')
for nome, info in DATASETS.items():
    df = info['df']
    print(f'  {nome:<22}: {info["path"].name}  ({len(df)} registros)')

# Tabela comparativa
print('\n[Tabela comparativa de métricas]')
print(f"  {'Modelo/Dataset':<34} {'acc':>6} {'f1_mac':>7} {'f1_0':>6} {'f1_1':>6} {'roc':>7} {'pr':>7}")
print('  ' + '-' * 75)

# Referências
print(f"  {'[REF] V1 (logreg v1)':<34} {V1_REF['accuracy']:>6.4f} {V1_REF['f1_macro']:>7.4f} {'N/A':>6} {'N/A':>6} {V1_REF['roc_auc']:>7.4f} {'N/A':>7}")
print(f"  {'[REF] V2 balanced (logreg v2)':<34} {V2_REF['accuracy']:>6.4f} {V2_REF['f1_macro']:>7.4f} {'N/A':>6} {'N/A':>6} {V2_REF['roc_auc']:>7.4f} {'N/A':>7}")

for chave, res in RESULTADOS.items():
    m = res['metricas']
    label = f"{res['ds_nome']}__{res['mod_nome']}"
    roc   = f"{m['roc_auc']:.4f}" if m['roc_auc'] is not None else 'N/A'
    pr    = f"{m['pr_auc']:.4f}"  if m['pr_auc']  is not None else 'N/A'
    print(f"  {label:<34} {m['accuracy']:>6.4f} {m['f1_macro']:>7.4f} {m['f1_label_0']:>6.4f} {m['f1_label_1']:>6.4f} {roc:>7} {pr:>7}")

# Melhor geral
melhor_geral = max(RESULTADOS.keys(), key=lambda k: RESULTADOS[k]['metricas']['f1_macro'])
melhor_v3_bal_chave  = max([k for k in RESULTADOS if k.startswith('v3_balanced__')],
                            key=lambda k: RESULTADOS[k]['metricas']['f1_macro'])
melhor_v3_tc_chave   = max([k for k in RESULTADOS if k.startswith('v3_text_control__')],
                            key=lambda k: RESULTADOS[k]['metricas']['f1_macro'])

print(f'\n[Melhor modelo geral]              {melhor_geral} — f1_macro={RESULTADOS[melhor_geral]["metricas"]["f1_macro"]}')
print(f'[Melhor modelo V3 balanced]        {melhor_v3_bal_chave} — f1_macro={RESULTADOS[melhor_v3_bal_chave]["metricas"]["f1_macro"]}')
print(f'[Melhor modelo V3 text_control]    {melhor_v3_tc_chave} — f1_macro={RESULTADOS[melhor_v3_tc_chave]["metricas"]["f1_macro"]}')

# Comparação V2 vs V3 balanced (mesmo modelo)
print('\n[Comparação V2 vs V3 balanced — mesmo modelo logreg]')
chaves_logreg_v2  = 'v2_balanced__logreg'
chaves_logreg_v3  = 'v3_balanced__logreg'
m_v2 = RESULTADOS[chaves_logreg_v2]['metricas']
m_v3 = RESULTADOS[chaves_logreg_v3]['metricas']
for met in ['accuracy', 'f1_macro', 'roc_auc']:
    v2_val = m_v2[met] if m_v2[met] is not None else float('nan')
    v3_val = m_v3[met] if m_v3[met] is not None else float('nan')
    delta = v3_val - v2_val if not (np.isnan(v2_val) or np.isnan(v3_val)) else float('nan')
    tendencia = ('MELHORA' if delta > 0.005 else ('QUEDA' if delta < -0.005 else 'ESTÁVEL'))
    print(f'  {met:<18}: V2={v2_val:.4f}  V3={v3_val:.4f}  delta={delta:+.4f}  [{tendencia}]')

# Alerta dependência Fake.Br
print('\n[Alerta — dependência de dataset_origem na V3]')
if registros_por_origem:
    df_orig = pd.DataFrame(registros_por_origem)
    for chave in [melhor_v3_bal_chave]:
        sub = df_orig[df_orig['chave_modelo'] == chave].sort_values('f1_macro')
        if len(sub) > 0:
            f1_min  = sub['f1_macro'].min()
            f1_max  = sub['f1_macro'].max()
            origem_pior  = sub.iloc[0]['dataset_origem']
            origem_melhor = sub.iloc[-1]['dataset_origem']
            gap = f1_max - f1_min
            print(f'  Modelo: {chave}')
            print(f'  Melhor origem: {origem_melhor} (f1={f1_max:.4f})')
            print(f'  Pior origem  : {origem_pior} (f1={f1_min:.4f})')
            print(f'  Gap f1_macro entre origens: {gap:.4f}')
            if gap > 0.10:
                print('  *** ALERTA: Gap > 0.10 — possível dependência de dataset_origem. ***')
            else:
                print('  OK: desempenho relativamente estável entre origens.')

print(SEP)

RELATÓRIO FINAL — TREINO BASELINE V3 COMPARATIVO

[Datasets utilizados]
  v2_balanced           : dataset_final_treino_v2_balanced_2026-05-19_00-48-15.csv  (508 registros)
  v3_balanced           : dataset_final_treino_v3_balanced_2026-05-26_23-54-08.csv  (10340 registros)
  v3_text_control       : dataset_final_treino_v3_text_control_2026-05-26_23-54-08.csv  (9050 registros)

[Tabela comparativa de métricas]
  Modelo/Dataset                        acc  f1_mac   f1_0   f1_1     roc      pr
  ---------------------------------------------------------------------------
  [REF] V1 (logreg v1)               0.8625  0.8614    N/A    N/A  0.9359     N/A
  [REF] V2 balanced (logreg v2)      0.9020  0.9018    N/A    N/A  0.9708     N/A
  v2_balanced__logreg                0.9020  0.9019 0.9038 0.9000  0.9727  0.9721
  v2_balanced__svm                   0.9118  0.9117 0.9091 0.9143  0.9681  0.9687
  v2_balanced__nb                    0.8627  0.8627 0.8627 0.8627  0.9639  0.9657
  v3_balanced__lo

## Seção 14 — Recomendação de candidato principal

In [17]:
SEP = '=' * 72
print(SEP)
print('RECOMENDAÇÃO DE MODELO CANDIDATO PRINCIPAL')
print(SEP)

# Comparação V3_balanced vs V3_text_control (logreg)
m_v3tc = RESULTADOS['v3_text_control__logreg']['metricas']
delta_bal_vs_tc = m_v3['f1_macro'] - m_v3tc['f1_macro']

print(f"""
Critérios considerados:
  1. F1 macro geral no conjunto de teste
  2. Estabilidade por dataset_origem (V2_PROPRIA / FAKETRUEBR / FAKEBR_TEXT_NORMALIZED)
  3. Estabilidade por faixa de tamanho
  4. Impacto do controle de viés (V3_balanced vs V3_text_control)

V3 balanced — melhor logreg:
  accuracy  = {m_v3['accuracy']}
  f1_macro  = {m_v3['f1_macro']}
  roc_auc   = {m_v3['roc_auc'] if m_v3['roc_auc'] else 'N/A'}

V3 text_control — melhor logreg:
  accuracy  = {m_v3tc['accuracy']}
  f1_macro  = {m_v3tc['f1_macro']}
  roc_auc   = {m_v3tc['roc_auc'] if m_v3tc['roc_auc'] else 'N/A'}

Delta f1_macro (balanced - text_control): {delta_bal_vs_tc:+.4f}
""")

if delta_bal_vs_tc > 0.01:
    print('Interpretação: V3_balanced supera text_control em F1 macro.')
    print('  Isso pode indicar que o campo texto_principal_modelo já reduzia o viés')
    print('  adequadamente, e o text_control perde informação textual útil.')
elif delta_bal_vs_tc < -0.01:
    print('Interpretação: V3_text_control supera balanced em F1 macro.')
    print('  Isso sugere que o viés de tamanho estava prejudicando a V3_balanced.')
    print('  Considerar V3_text_control como candidato principal.')
else:
    print('Interpretação: desempenho equivalente entre balanced e text_control.')
    print('  Preferir V3_balanced por maior volume de treino.')

print(f'\nModelo candidato principal para a Fase 8: {melhor_v3_bal_chave}')
print(f'  Arquivo: baseline_tfidf_{melhor_v3_bal_chave.split("__")[1]}_v3_balanced_{TS}.joblib')
print(SEP)

RECOMENDAÇÃO DE MODELO CANDIDATO PRINCIPAL

Critérios considerados:
  1. F1 macro geral no conjunto de teste
  2. Estabilidade por dataset_origem (V2_PROPRIA / FAKETRUEBR / FAKEBR_TEXT_NORMALIZED)
  3. Estabilidade por faixa de tamanho
  4. Impacto do controle de viés (V3_balanced vs V3_text_control)

V3 balanced — melhor logreg:
  accuracy  = 0.94
  f1_macro  = 0.94
  roc_auc   = 0.9787

V3 text_control — melhor logreg:
  accuracy  = 0.9243
  f1_macro  = 0.9243
  roc_auc   = 0.9749

Delta f1_macro (balanced - text_control): +0.0157

Interpretação: V3_balanced supera text_control em F1 macro.
  Isso pode indicar que o campo texto_principal_modelo já reduzia o viés
  adequadamente, e o text_control perde informação textual útil.

Modelo candidato principal para a Fase 8: v3_balanced__svm
  Arquivo: baseline_tfidf_svm_v3_balanced_2026-05-27_00-29-53.joblib


## Seção 15 — Conclusão metodológica

In [18]:
SEP = '=' * 72
print(SEP)
print('CONCLUSÃO METODOLÓGICA — FASE 7')
print(SEP)
print("""
A Fase 7 realizou o treino e comparação de modelos clássicos de NLP (TF-IDF
combinado com Logistic Regression, LinearSVM e Multinomial Naive Bayes) sobre
três versões de dataset: V2 balanced, V3 balanced e V3 text_control.

Pontos metodológicos relevantes:

1. AMPLIAÇÃO DE VOLUME
   A V3 triplicou o volume em relação à V2 (de ~508 para ~10.340 registros
   balanceados), incorporando FakeTrueBR e Fake.Br text_normalized além da
   base própria do projeto. A comparação V2 vs V3 mede o impacto direto dessa
   ampliação no desempenho dos modelos clássicos.

2. CONTROLE DE VIÉS DE TAMANHO
   O campo texto_principal_modelo foi usado como texto de treino preferencial,
   pois trunca o Fake.Br ao segmento principal, reduzindo artificialmente a
   vantagem de textos reais mais longos. A versão text_control vai além,
   selecionando apenas registros dentro de faixas de tamanho compatíveis entre
   classes. A comparação V3_balanced vs V3_text_control isola o efeito do
   viés de tamanho residual.

3. ANÁLISE POR ORIGEM
   A verificação de métricas separadas por dataset_origem (V2_PROPRIA,
   FAKETRUEBR, FAKEBR_TEXT_NORMALIZED) é essencial para detectar dependência
   do modelo de características específicas de um único corpus. Um modelo que
   performa bem globalmente mas falha na V2_PROPRIA (base do projeto real) é
   metodologicamente fraco para o objetivo do TCC.

4. CRITÉRIO DE SELEÇÃO DO MODELO
   O candidato principal deve considerar não apenas o F1 macro geral, mas:
   a) Estabilidade entre origens de dados;
   b) Robustez a variações de comprimento de texto;
   c) Desempenho equilibrado entre label=0 (fake) e label=1 (real),
      evitando viés sistemático para uma das classes.

5. LIMITAÇÕES REMANESCENTES
   Os modelos clássicos TF-IDF não capturam semântica profunda nem dependências
   contextuais longas. A próxima fase natural (Fase 8) pode avaliar modelos
   transformer (BERTimbau) para comparação com esta baseline estabelecida.
   Os modelos aqui treinados servem como limite inferior de desempenho esperado
   de qualquer arquitetura mais sofisticada.
""")
print(SEP)
print(f'Notebook executado em: {TS}')
print('Todos os artefatos salvos com timestamp. Nenhum arquivo existente foi alterado.')
print(SEP)

CONCLUSÃO METODOLÓGICA — FASE 7

A Fase 7 realizou o treino e comparação de modelos clássicos de NLP (TF-IDF
combinado com Logistic Regression, LinearSVM e Multinomial Naive Bayes) sobre
três versões de dataset: V2 balanced, V3 balanced e V3 text_control.

Pontos metodológicos relevantes:

1. AMPLIAÇÃO DE VOLUME
   A V3 triplicou o volume em relação à V2 (de ~508 para ~10.340 registros
   balanceados), incorporando FakeTrueBR e Fake.Br text_normalized além da
   base própria do projeto. A comparação V2 vs V3 mede o impacto direto dessa
   ampliação no desempenho dos modelos clássicos.

2. CONTROLE DE VIÉS DE TAMANHO
   O campo texto_principal_modelo foi usado como texto de treino preferencial,
   pois trunca o Fake.Br ao segmento principal, reduzindo artificialmente a
   vantagem de textos reais mais longos. A versão text_control vai além,
   selecionando apenas registros dentro de faixas de tamanho compatíveis entre
   classes. A comparação V3_balanced vs V3_text_control isola o efeit